## Imports

In [23]:
!pip install sentence-transformers faiss-cpu

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import time 
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Metadata Enrich
- PDF name, type add

In [2]:
def add_metadata(doc, pdf_path):
    doc.metadata["source_file"] = pdf_path.name
    doc.metadata["file_type"] = "pdf"
    return doc

## Domain Tagging

- legal / medical detect

In [3]:
def detect_domain(text):
    text = text.lower()
    if any(k in text for k in ["patient", "diagnosis", "treatment"]):
        return "medical"
    if any(k in text for k in ["agreement", "tenant", "clause"]):
        return "legal"
    return "unknown"

## Text Cleaning

- spaces, line breaks clean

In [4]:
def clean_text(text):
    return " ".join(text.split())

## Filtering Pages

- empty / useless pages remove

In [5]:
def is_valid_page(text, min_len=50):
    return len(text.strip()) >= min_len

## Structure Detect

- clause / report type

In [6]:
def deduplicate_docs(docs):
    seen = set()
    unique = []
    for d in docs:
        if d.page_content not in seen:
            unique.append(d)
            seen.add(d.page_content)
    return unique

## Deduplication

- repeated pages remove

In [7]:
def detect_language(text):
    try:
        text.encode("ascii")
        return "en"
    except:
        return "non-en"

Language Detect

👉 simple english detect

In [8]:
def detect_language(text):
    try:
        text.encode("ascii")
        return "en"
    except:
        return "non-en"

In [9]:
def load_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")

    for pdf_file in pdf_files:
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()

        print(pdf_file.name, "pages:", len(documents))

        all_documents.extend(documents)

    return all_documents


all_pdf_documents = load_pdfs(r"C:\Users\HP\OneDrive\Desktop\project\AI\AI-Powered RAG System\data")
print("Total pages:", len(all_pdf_documents))

Found 6 PDF files
contract1.pdf pages: 53
contract2.pdf pages: 16
contract3.pdf pages: 26
Hospital-Discharge-Summary-Format.pdf pages: 2
MedicalReport1.pdf pages: 4
Sample-Lab-Report.pdf pages: 3
Total pages: 104


In [10]:
print(type(all_pdf_documents))
print(type(all_pdf_documents[0]))
print(all_pdf_documents[0].metadata)

<class 'list'>
<class 'langchain_core.documents.base.Document'>
{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2021-06-21T11:04:58-07:00', 'author': 'Admin', 'moddate': '2021-06-21T11:04:58-07:00', 'source': 'C:\\Users\\HP\\OneDrive\\Desktop\\project\\AI\\AI-Powered RAG System\\data\\sample_legal\\contract1.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1'}


In [11]:
processed_docs = []

for d in all_pdf_documents:
    text = clean_text(d.page_content)

    if not is_valid_page(text):
        continue

    d.page_content = text
    d.metadata["domain"] = detect_domain(text)

    processed_docs.append(d)

print("After processing:", len(processed_docs))

After processing: 104


In [12]:
print(processed_docs[0].metadata)

{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2021-06-21T11:04:58-07:00', 'author': 'Admin', 'moddate': '2021-06-21T11:04:58-07:00', 'source': 'C:\\Users\\HP\\OneDrive\\Desktop\\project\\AI\\AI-Powered RAG System\\data\\sample_legal\\contract1.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'domain': 'legal'}


In [13]:
print(all_pdf_documents[0].metadata)

{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2021-06-21T11:04:58-07:00', 'author': 'Admin', 'moddate': '2021-06-21T11:04:58-07:00', 'source': 'C:\\Users\\HP\\OneDrive\\Desktop\\project\\AI\\AI-Powered RAG System\\data\\sample_legal\\contract1.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'domain': 'legal'}


In [14]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2021-06-21T11:04:58-07:00', 'author': 'Admin', 'moddate': '2021-06-21T11:04:58-07:00', 'source': 'C:\\Users\\HP\\OneDrive\\Desktop\\project\\AI\\AI-Powered RAG System\\data\\sample_legal\\contract1.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'domain': 'legal'}, page_content='1 THE INDIAN CONTRACT ACT, 1872 ____________ ARRANGEMENT OF SECTIONS ____________ SECTIONS PREAMBLE PRELIMINARY 1. Short title. Extent. Commencement. Saving. 2. Interpretation-clause. CHAPTER I OF THE COMMUNICATION, ACCEPTANCE AND REVOCATION OF PROPOSALS 3. Communication, acceptance and revocation of proposals. 4. Communication when complete. 5. Revocation of proposals and acceptances. 6. Revocation how made. 7. Acceptance must be absolute. 8. Acceptance by performing conditions, or receiving consideration. 9. Promises, express and implied. CHAPTER II OF CONTRACTS, VOIDABLE CONTRACTS AND VOID AGREE

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text

# default stopwords
stop_words = set(text.ENGLISH_STOP_WORDS)

# add custom noisy words
custom_noise = {
    "000","ibid","rep","sub","shall","may","said",
    "thereof","therein","hereby","herein","thereby",
    "does","make"
}

stop_words = stop_words.union(custom_noise)

texts = [doc.page_content for doc in all_pdf_documents if doc.page_content.strip()]

vectorizer = TfidfVectorizer(
    stop_words=list(stop_words),
    max_features=40,
    ngram_range=(1,2),
    token_pattern=r"\b[a-zA-Z]{3,}\b"   # only words >=3 letters
)

X = vectorizer.fit_transform(texts)
keywords = vectorizer.get_feature_names_out()

print("Clean Keywords:")
for k in keywords:
    print(k)

Clean Keywords:
act
agent
agreement
authority
bailee
business
buyer
contract
contracts
delivery
effect
entitled
firm
goods
liable
notice
partner
partners
partnership
party
pay
payment
performance
person
price
principal
promise
property
repealed
repealed repealed
right
rupees
sale
schedule
section
sell
seller
surety
time
void


In [1]:
file_texts = defaultdict(list)

for doc in all_pdf_documents:
    source = doc.metadata.get("source", "unknown")
    file_texts[source].append(doc.page_content)

for file, texts in file_texts.items():
    vectorizer = TfidfVectorizer(stop_words="english", max_features=10)
    X = vectorizer.fit_transform(texts)
    keywords = vectorizer.get_feature_names_out()
    
    print(f"\n{file}:")
    print(", ".join(keywords))

NameError: name 'defaultdict' is not defined

In [17]:
sizes = [600, 1000, 1400]

for size in sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.2)
    )

    chunks = splitter.split_documents(all_pdf_documents)

    print(f"Chunk size {size} → total chunks:", len(chunks))

Chunk size 600 → total chunks: 656
Chunk size 1000 → total chunks: 409
Chunk size 1400 → total chunks: 292


In [18]:
def is_valid_chunk(text, min_len=100):
    return len(text.strip()) >= min_len

def deduplicate_chunks(chunks):
    seen = set()
    unique = []
    for c in chunks:
        if c.page_content not in seen:
            unique.append(c)
            seen.add(c.page_content)
    return unique

def chunk_documents_advanced(documents):
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    
    chunks = splitter.split_documents(documents)
    
    # filter
    chunks = [c for c in chunks if is_valid_chunk(c.page_content)]
    
    # dedup
    chunks = deduplicate_chunks(chunks)
    
    return chunks

In [19]:
all_chunks = chunk_documents_advanced(all_pdf_documents)

print("Total chunks:", len(all_chunks))
print(all_chunks[0].metadata)

Total chunks: 409
{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2021-06-21T11:04:58-07:00', 'author': 'Admin', 'moddate': '2021-06-21T11:04:58-07:00', 'source': 'C:\\Users\\HP\\OneDrive\\Desktop\\project\\AI\\AI-Powered RAG System\\data\\sample_legal\\contract1.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'domain': 'legal'}


## Load MiniLM Model

In [20]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embedding_dim = model.get_sentence_embedding_dimension()
print("Embedding dimension:", embedding_dim)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 404.80it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


In [21]:
text =[c.page_content for c in all_chunks]
metadata=[c.metadata for c in all_chunks]

print("Total chunks:",len(text))

Total chunks: 409


In [22]:
start = time.time()

embeddings = model.encode(
    text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

end = time.time()

print("Embedding shape:", embeddings.shape)
print("Time taken:", end-start)

Batches: 100%|██████████| 13/13 [00:33<00:00,  2.56s/it]

Embedding shape: (409, 384)
Time taken: 33.38716459274292


In [23]:
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("FAISS index size",index.ntotal)

FAISS index size 409


In [24]:
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000002273FF803F0> >

In [25]:
print(embeddings[0])

[-2.02019233e-02  5.74038103e-02  2.37054843e-03 -5.19026406e-02
 -1.28505081e-01  6.79149330e-02  3.43343206e-02 -6.63407962e-04
  7.90819619e-03  5.55481985e-02 -8.07579514e-03 -6.51466241e-03
  4.65828292e-02 -8.21668003e-03  6.48876429e-02  2.72168629e-02
  5.96396998e-02 -1.62992943e-02 -6.89575896e-02  6.91856295e-02
  8.87575224e-02  6.42130822e-02  2.35936996e-02  1.13243550e-04
  7.96841923e-03 -4.82003205e-02  3.00781075e-02 -3.22501361e-02
  9.69411992e-03 -2.62279119e-02 -2.84553934e-02  6.85341656e-02
  3.93749923e-02  5.98311760e-02  1.56259462e-02 -1.52314575e-02
 -2.72335466e-02 -6.92229941e-02  4.35804240e-02 -6.49929941e-02
 -3.83878052e-02  1.70471165e-02  1.08183566e-02 -4.31821533e-02
 -7.60379387e-03 -2.14292854e-02 -3.36874351e-02  5.74665703e-03
 -1.13953508e-01 -8.39383155e-03  3.24088777e-03 -1.23062572e-02
 -3.64053547e-02  4.64715436e-02 -2.83914749e-02 -8.46523568e-02
 -2.80663241e-02 -5.16817607e-02  1.27434998e-03  3.16169709e-02
  3.19655053e-02  5.72140

In [26]:
vec0 = index.reconstruct(0)
print(vec0)

[-2.02019233e-02  5.74038103e-02  2.37054843e-03 -5.19026406e-02
 -1.28505081e-01  6.79149330e-02  3.43343206e-02 -6.63407962e-04
  7.90819619e-03  5.55481985e-02 -8.07579514e-03 -6.51466241e-03
  4.65828292e-02 -8.21668003e-03  6.48876429e-02  2.72168629e-02
  5.96396998e-02 -1.62992943e-02 -6.89575896e-02  6.91856295e-02
  8.87575224e-02  6.42130822e-02  2.35936996e-02  1.13243550e-04
  7.96841923e-03 -4.82003205e-02  3.00781075e-02 -3.22501361e-02
  9.69411992e-03 -2.62279119e-02 -2.84553934e-02  6.85341656e-02
  3.93749923e-02  5.98311760e-02  1.56259462e-02 -1.52314575e-02
 -2.72335466e-02 -6.92229941e-02  4.35804240e-02 -6.49929941e-02
 -3.83878052e-02  1.70471165e-02  1.08183566e-02 -4.31821533e-02
 -7.60379387e-03 -2.14292854e-02 -3.36874351e-02  5.74665703e-03
 -1.13953508e-01 -8.39383155e-03  3.24088777e-03 -1.23062572e-02
 -3.64053547e-02  4.64715436e-02 -2.83914749e-02 -8.46523568e-02
 -2.80663241e-02 -5.16817607e-02  1.27434998e-03  3.16169709e-02
  3.19655053e-02  5.72140

In [27]:
vec = index.reconstruct(0)
print(len(vec))

384


In [29]:
import faiss
import os

os.makedirs("outputs/faiss_indexes", exist_ok=True)

faiss.write_index(index, "outputs/faiss_indexes/combined.faiss")

print("FAISS index saved")

FAISS index saved


In [30]:
import os
print(os.listdir("outputs/faiss_indexes"))

['combined.faiss', 'metadata.pkl', 'text.pkl']


In [31]:
faiss.write_index(index, "outputs/faiss_indexes/combined.faiss")